# AgentRegistry, end to end: scaffold a dice agent → run it on kagent **and** AWS Bedrock AgentCore

**Persona:** a **developer**. Run one cell at a time like a terminal — see the output, watch the project appear in the Explorer. We scaffold a dice agent with `arctl`, run it locally, publish it, then deploy **the same agent** to two runtimes — Solo Enterprise for **kagent** (local kind) and **AWS Bedrock AgentCore** — by changing one line, the Deployment's `runtimeRef`.

> Engineer setup (kind + kagent + daemon) is one-time and done before this — see the README. Pick the **Bash** kernel (or connect to the Jupyter server per the README).

## Connect to the platform

In [ ]:
source scripts/connect.sh

## 1. Create a new agent project

Scaffolds the dice-rolling agent into `agentdemo/`. **Open it in the Explorer** to walk through `roll_die` / `check_prime`.

In [ ]:
arctl init agent agentdemo --framework adk --language python --model-provider anthropic --model-name claude-haiku-4-5

## 2. Walk through the dice agent

In [ ]:
cat agentdemo/agentdemo/agent.py

## 3. Build the agent image

In [ ]:
arctl build ./agentdemo

Run it locally in an interactive chat (use a **terminal** — it's interactive):

```sh
arctl run ./agentdemo
# Roll a 20-sided die and tell me if the result is prime.
```

## 4. Publish to the catalog

In [ ]:
arctl build ./agentdemo --push

In [ ]:
arctl apply -f agentdemo/agent.yaml

In [ ]:
arctl get agent agentdemo

## 5. Deploy the agent onto kagent (runtime #1)

The `kind-kagent` Runtime was registered during setup. This binds the agent to it — `runtimeRef: kind-kagent` is **the one line that changes for AWS later.**

In [ ]:
envsubst < yaml/deploy-kagent.yaml | arctl apply -f -

In [ ]:
arctl get deployments

## 6. Talk to the dice agent — through real OIDC

`ask.sh` mints a real Keycloak token for **alice** and sends an A2A message. Watch it call `roll_die` then `check_prime`.

In [ ]:
./scripts/ask.sh "Roll a 20-sided die and tell me whether the result is a prime number."

---
# The punchline: the **same** agent on AWS Bedrock AgentCore (runtime #2)

The identical agent, deployed to AWS — native Bedrock Claude via the AWS role (no API key). **Needs an AWS account; skip for a local-only demo.**

## 7. Sign in to AWS

In [ ]:
source scripts/aws-login.sh

## 8. Deploy the same agent to AgentCore

One command: makes the agent multi-cloud, grants the cross-account role, registers the `BedrockAgentCore` runtime, pushes image+source, and deploys. Waits for READY.

In [ ]:
./scripts/agentcore-deploy.sh

## 9. Test the dice agent on AgentCore

In [ ]:
./scripts/ac-invoke.sh "Roll a 20-sided die and tell me whether the result is a prime number."

**The takeaway:** one agent, scaffolded with `arctl`, published once, ran unchanged on Kubernetes (kagent) *and* AWS Bedrock AgentCore.

## Reset / teardown

```sh
./scripts/reset.sh      # back to start (clears agentdemo/, deployments, AWS); platform stays up
./scripts/cleanup.sh    # full teardown
```